In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
from ast import literal_eval

import iqpopt as iqp
import iqpopt.gen_qml as gen
import json 

In [ ]:
def bootstrap_mmd2(
    experiment_samples,
    rot_angles,
    B=500,
    R=1,
    n_iqpopt_samples=10000,
    seed=0,
):
    """
    Bootstrap uncertainty for IQPopt-estimated MMD^2.

    experiment_samples: array of shape (N, n_qubits)
    rot_angles: dictionary mapping gate strings to angles
    B: number of bootstrap replicates
    R: number of IQPopt MC repetitions per bootstrap replicate
    """

    rng = np.random.default_rng(seed)

    # Build IQPopt simulator
    gates = []
    params = []
    num_qubits = 0

    for k, angle_raw in rot_angles.items():
        angle = angle_raw / 2
        params.append(angle)

        gate = literal_eval(k)

        if isinstance(gate, int):
            qubits = [gate]
        else:
            qubits = list(gate)

        gates.append([qubits])
        num_qubits = max(num_qubits, max(qubits) + 1)

    circuit = iqp.IqpSimulator(num_qubits, gates)
    params_jax = jnp.asarray(params)

    num_samples = experiment_samples.shape[0]

    # Pre-generate JAX keys
    master_key = jax.random.PRNGKey(seed)
    keys = jax.random.split(master_key, B * R)

    boot_means = []
    all_values = []

    key_counter = 0

    for b in range(B):
        # Bootstrap sample: same size as original, sampled with replacement
        bs_index = rng.integers(0, num_samples, size=num_samples)
        bs_samples = experiment_samples[bs_index]
        bs_samples_jax = jnp.asarray(bs_samples)

        replicate_values = []

        for r in range(R):
            key = keys[key_counter]
            key_counter += 1

            loss = gen.mmd_loss_iqp(
                params_jax,
                circuit,
                bs_samples_jax,
                sigma=[1.0/np.sqrt(2), num_qubits / (4*np.sqrt(2)), num_qubits / (2*np.sqrt(2))],
                n_ops=n_iqpopt_samples,
                n_samples=n_iqpopt_samples,
                key=key,
            )

            loss_float = float(loss)
            replicate_values.append(loss_float)
            all_values.append(loss_float)

        # Average over IQPopt MC repetitions for this bootstrap sample
        boot_means.append(np.mean(replicate_values))

    boot_means = np.asarray(boot_means)
    all_values = np.asarray(all_values)

    mean = np.mean(boot_means)
    se = np.std(boot_means, ddof=1)
    ci95 = np.percentile(boot_means, [2.5, 97.5])

    return {
        "mean": mean,
        "bootstrap_se": se,
        "ci95": ci95,
        "boot_values": boot_means,
        "all_values": all_values,
    }

In [ ]:
with open('./ibm_results/large_circuit_rot_angles.json', 'r') as f:
    rot_angles = json.load(f)

result_self = bootstrap_mmd2(
    np.load('./ibm_results/percolation_samples.npy'), # Change
    rot_angles,
    B=500,
    R=5,
    n_iqpopt_samples=5000,
    seed=0,
)
result_self